# 03 — Exploration

This notebook provides a concise analytical exploration of the merged European dataset combining educational learning indicators and EF English proficiency measures.

The objective is not to perform exhaustive exploratory analysis, but rather to identify the main empirical patterns, validate the analytical consistency of the merged dataset, and prepare the visual logic developed in the subsequent dashboard-oriented notebook.

In [ ]:
# 1. Imports and visualization helper
import pathlib
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Plotly defaults for publication style
px.defaults.template = "plotly_white"

def apply_standard_layout(fig, height=480, margin_b=40):
    """Apply consistent publication-style layout to plotly figures."""
    fig.update_layout(
        title_x=0.5,
        margin=dict(l=40, r=40, t=60, b=margin_b),
        height=height,
        font=dict(size=11)
    )
    return fig

In [ ]:
# 2. Dataset overview
project_root = pathlib.Path().resolve().parent
data_path = project_root / 'data' / 'processed' / 'merged_analytical.csv'
df = pd.read_csv(data_path)

print('✓ Loaded:', data_path)
print('✓ Shape:', df.shape)
print('✓ Year coverage:', int(df['year'].min()), 'to', int(df['year'].max()))
print('✓ Unique countries (iso3):', df['iso3'].nunique())
print('✓ Duplicate iso3-year pairs:', df.duplicated(subset=['iso3','year']).sum())

✓ Loaded: C:\Users\henri_ugzoq54\OneDrive\Workspace\2025_2026\Trento M1 DS\Semester 2\Data Vis\Project\Project Visualisation EF\data\processed\merged_analytical.csv
✓ Shape: (365, 7)
✓ Year coverage: 2012 to 2024
✓ Unique countries (iso3): 34
✓ Duplicate iso3-year pairs: 0


## 2. Data quality overview

Before investigating the main empirical patterns, the merged dataset is evaluated in terms of completeness, consistency, and EF coverage across countries and years.

This preliminary assessment helps identify the proportion of usable observations and provides context for interpreting subsequent visualizations and comparative analyses.

In [12]:
# Missing values
missing = df.isnull().sum().rename('missing').to_frame()
missing['pct'] = (missing['missing'] / len(df) * 100).round(1)
display(missing[['missing','pct']])

complete_cases = df.dropna().shape[0]
ef_match_rate = df['ef_percentile'].notna().sum() / len(df)
print(f'\nCompleteness: {complete_cases}/{len(df)} complete rows ({complete_cases/len(df)*100:.1f}%)')
print(f'EF match rate: {ef_match_rate:.1%}')

,missing,pct
geo,0,0.0
iso3,0,0.0
year,0,0.0
learning,0,0.0
learning_percentile,0,0.0
ef_percentile,200,54.8
gap_pct,200,54.8



Completeness: 165/365 complete rows (45.2%)
EF match rate: 45.2%


**Interpretation:** Missing `ef_percentile` and `gap_pct` values mainly occur when EF observations are unavailable for specific country-year combinations due to temporal coverage limitations and lag alignment constraints. Comparative interpretations should therefore be made cautiously for countries with limited longitudinal coverage.

## 3. Core gap analysis

The `gap_pct` indicator measures the difference between EF English proficiency percentiles and learning exposure percentiles across European countries and years.

Positive values indicate countries achieving higher English proficiency than expected relative to their learning exposure indicators, while negative values suggest lower proficiency outcomes relative to the observed educational exposure context.

The following distribution analysis provides an overview of the overall heterogeneity and imbalance structure of the European dataset.

In [13]:
# Descriptive statistics
stats = df['gap_pct'].describe().round(3)
display(stats.to_frame(name='gap_pct'))

# Histogram + boxplot
fig = px.histogram(
    df,
    x='gap_pct',
    nbins=24,
    marginal='box',
    title='Distribution of EF proficiency gap',
    color_discrete_sequence=['#2E8B57']  # green
)

# Neutral parity line
fig.add_vline(
    x=0,
    line_dash='dash',
    line_color='black',
    opacity=0.7
)

apply_standard_layout(fig, height=420)

fig.show()

,gap_pct
count,165.000
mean,0.158
std,0.394
min,-0.733
25%,-0.100
50%,0.085
75%,0.528
max,0.900


The distribution of the EF proficiency gap highlights a substantial heterogeneity across European countries and years. While the distribution is slightly centered above zero, indicating that several countries tend to achieve higher English proficiency than expected relative to their learning exposure indicators, the overall dispersion remains large.

This variability suggests that educational exposure alone is insufficient to explain observed proficiency outcomes. Structural, cultural, economic, and linguistic environments likely contribute to significant differences in English proficiency performance across Europe.

The presence of both strongly positive and strongly negative gaps supports the relevance of a comparative and visualization-oriented approach. Rather than converging toward a common European pattern, countries appear to follow distinct proficiency trajectories, reinforcing the importance of spatial and temporal analysis in the remainder of the project.

The dashed vertical line at zero represents the theoretical parity threshold between expected learning exposure and observed EF proficiency outcomes.

## 4. Temporal evolution

The temporal evolution of the average `gap_pct` provides a high-level view of the stability of English proficiency disparities across Europe over time.

Rather than focusing on individual country trajectories, this aggregate perspective helps identify whether the overall European pattern tends toward convergence, persistence, or increasing divergence between learning exposure and observed proficiency outcomes.

In [14]:
# Yearly aggregation
yearly = (
    df.groupby('year', as_index=False)['gap_pct']
    .mean()
)

fig = px.line(
    yearly,
    x='year',
    y='gap_pct',
    markers=True,
    title='Temporal evolution of the EF proficiency gap'
)

# Semantic color
fig.update_traces(
    line=dict(color='#2E8B57', width=3),
    marker=dict(size=7, color='#2E8B57')
)

# Neutral reference line
fig.add_hline(
    y=0,
    line_dash='dash',
    line_color='black',
    opacity=0.6
)

apply_standard_layout(fig, height=430)

fig.update_xaxes(dtick=1)

fig.show()

The average EF proficiency gap remains relatively stable across the observed period, with no clear convergence toward the parity threshold. While moderate fluctuations are visible between years, the overall pattern suggests that structural differences between countries persist over time rather than disappearing.

The consistently positive average gap indicates that, at the European aggregate level, observed English proficiency tends to remain slightly higher than expected relative to the underlying learning exposure indicators.

This temporal stability reinforces the hypothesis that long-term educational, cultural, and linguistic environments play a persistent role in shaping proficiency outcomes across Europe.

## 5. European spatial patterns

The following choropleth map provides a spatial overview of the average EF proficiency gap across European countries.

By aggregating country-level differences over time, the visualization highlights persistent regional contrasts between observed English proficiency outcomes and learning exposure indicators. The objective is not to rank countries exhaustively, but rather to identify broad geographic structures and spatial heterogeneity within the European landscape.

In [20]:
# Choropleth map

fig_map = px.choropleth(
    country_gap,
    locations='iso3',
    color='gap_pct',

    scope='europe',
    projection='natural earth',

    color_continuous_scale=[
        [0.0, '#B22222'],   # negative gap
        [0.5, '#F5F5F5'],   # neutral
        [1.0, '#2E8B57']    # positive gap
    ],

    range_color=[-1, 1],

    title='European distribution of EF proficiency gaps',

    hover_name='iso3',

    labels={
        'gap_pct': 'Gap'
    }
)

fig_map.update_geos(
    showcountries=True,
    countrycolor='white',

    showcoastlines=True,
    coastlinecolor='lightgray',

    showland=True,
    landcolor='#F8F8F8',

    showframe=False
)

fig_map.update_layout(
    coloraxis_colorbar_title='Gap',

    margin=dict(
        l=0,
        r=0,
        t=60,
        b=0
    )
)

apply_standard_layout(fig_map, height=650)

fig_map.show()

The spatial distribution of the EF proficiency gap reveals persistent geographic heterogeneity across Europe. Northern and north-western European countries generally exhibit positive gaps, indicating higher observed English proficiency relative to learning exposure indicators, whereas several southern and eastern countries display lower relative outcomes.

These contrasts suggest that English proficiency performance cannot be explained solely by formal educational exposure. Broader structural factors — including cultural openness, media exposure, linguistic proximity, digital environments, and internationalization dynamics — may contribute to the observed regional disparities.

Rather than identifying a single European trajectory, the map highlights the coexistence of distinct spatial patterns and reinforces the relevance of a comparative visualization framework for analyzing language proficiency across Europe.

## 6. Key findings

- The EF proficiency gap displays substantial heterogeneity across European countries, with positive gaps slightly dominating the overall distribution.
- The average European gap remains relatively stable over time, suggesting persistent structural differences rather than convergence dynamics.
- Clear regional contrasts emerge across the continent, particularly between northern European countries with consistently positive gaps and several southern or eastern countries exhibiting lower relative proficiency outcomes.
- The spatial distribution of the gap suggests that English proficiency performance cannot be explained solely by formal educational exposure indicators.
- Dataset coverage remains partially constrained by EF temporal availability and lag alignment choices, requiring cautious interpretation of some country-level comparisons.